## RAG 모델 성능 평가하기

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [2]:
file_path = "../../data/Sustainability_report_2024_kr.pdf"

loader = PyPDFLoader(file_path)

docs = loader.load()
len(docs)
docs[:3]

[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.1 (Macintosh)', 'creationdate': '2024-11-25T11:10:32+09:00', 'moddate': '2024-11-25T11:10:46+09:00', 'trapped': '/False', 'source': '../../data/Sustainability_report_2024_kr.pdf', 'total_pages': 83, 'page': 0, 'page_label': '1'}, page_content='A Journey Towards  \na Sustainable Future\n삼성전자 지속가능경영보고서 2024'),
 Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.1 (Macintosh)', 'creationdate': '2024-11-25T11:10:32+09:00', 'moddate': '2024-11-25T11:10:46+09:00', 'trapped': '/False', 'source': '../../data/Sustainability_report_2024_kr.pdf', 'total_pages': 83, 'page': 1, 'page_label': '2'}, page_content='A Journey Towards  \na Sustainable Future\n삼성전자 지속가능경영보고서 2024\nCEO 메시지\n회사 소개\n이해관계자 소통\nOur Company\n04\n05\n06\n준법과 윤리경영\nPrinciple\n53\n중대성 평가\nMateriality Assessment\n08\n임직원\n공급망\n사회공헌\n개인정보보호/보안\n고객의 안전/품질\nPeople\n31\n39\n45\n48\n50\n경제성과\n사회성과\n환경성과\n지역별 수자원 현황

In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs)
len(chunks)

207

In [ ]:
# 벡터스토어 및 리트리버 구성
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate

embeddings = OpenAIEmbeddings()
persist_directory="../7_vectorstore/rag_eval_20"

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="samsung2024_eval",
)

In [ ]:
# 4. retriever 구성하기
retriever = vectorstore.as_retriever(
    search_kwargs = {"k" : 5}
)

In [6]:
#프롬프트 구성하기
from langchain_core.prompts import ChatPromptTemplate

system_template = """
    "You are a helpful assistant. Answer strictly based on the provided context. "
    "If the answer is not in the context, say you don't know."
    "context : {context}"
"""

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", system_template),
    ("human", "{question}"),
])

In [7]:
# 모델 구성하기
model = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

In [8]:
# 아웃풋 파서
from langchain_core.output_parsers import StrOutputParser
outputparser = StrOutputParser()

In [ ]:
# 8. 체인설정
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# 문서 합치는 함수
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

# 체인만들기
rag_chain = (
    {"context" : RunnableLambda(lambda x : x["question"]) | retriever | format_docs,
     "question" :RunnablePassthrough()
     }
     | rag_prompt
     | model
     | outputparser
)

In [11]:
rag_chain.invoke({"question": "삼성전자는 무슨일을 해?"})

'죄송하지만, 제공된 정보에는 삼성전자가 무슨 일을 하는지에 대한 내용이 없습니다.'

In [13]:
import pandas as pd
csv_path = "../../data/rag_eval.csv"
df = pd.read_csv(csv_path)
df

,user_input,reference_contexts,reference,synthesizer_name
0,What Samsung Electronics say about CSR D and E...,['Principle\nPlanet\nPeople\nCEO 메시지\nMessage ...,"Samsung Electronics explains that in 2023, glo...",single_hop_specific_query_synthesizer
1,한종희 삼성전자 부회장 지속가능경영에 대해 알려줘,"[""있어서는 비제조 분야 및 리스크 분석에 따라 제조 분야 2차 협력회사로 \n근로...",한종희 삼성전자 부회장은 지속가능경영을 삼성전자가 나아가야 할 방향의 흔들리지 않는...,single_hop_specific_query_synthesizer
2,Could you explain the role of the Device eXper...,['Principle\nPlanet\nPeople\n회사소개\nAbout Us\n삼...,The Device eXperience (DX) division at Samsung...,single_hop_specific_query_synthesizer
3,삼성닷컴 은 이해관계자 소통에서 어떤 역할을 하나요?,['Facts & Figures \nPrinciple\nPlanet\nPeople\...,"삼성닷컴은 고객과의 소통 채널 중 하나로, 제품과 서비스 품질, 안전한 제품 사용,...",single_hop_specific_query_synthesizer
4,What is the role of the 글로벌 구매 통합관리 시스템(G-SRM)...,['· 주주·투자자 의견 수렴\n 임직원\n· \x07안전하고 건강한 근로환경\n·...,The 글로벌 구매 통합관리 시스템(G-SRM) is part of the effo...,single_hop_specific_query_synthesizer
...,...,...,...,...
95,How did the company perform in terms of 폐전자제품 ...,['<1-hop>\n\n사내 폐기물 저감 실천 \n 폐제품 수거 체계 운영 상세내용...,"In 2021, the company collected 55.9 만 톤 of 폐전자...",multi_hop_specific_query_synthesizer
96,How did Samsung Electronics engage with intern...,['<1-hop>\n\n마련하고 유관 사업 활동을 분석하였습니다. 이후 각 활동과 ...,"In March 2024, Samsung Electronics conducted a...",multi_hop_specific_query_synthesizer
97,How DX부문 achieve 플래티넘 certification for 폐기물 매립...,['<1-hop>\n\n사내 폐기물 저감 실천 \n 폐제품 수거 체계 운영 상세내용...,DX부문 achieved the 플래티넘 certification for 폐기물 매...,multi_hop_specific_query_synthesizer
98,How DX부문 manage product and manufacturing haza...,['<1-hop>\n\n제품 및 제조과정 우려물질 관리\n∙ 제품 내 우려물질 및 ...,"In 2022년, DX부문 strengthened compliance and man...",multi_hop_specific_query_synthesizer


- user_input: 질문
- reference_contexts: 예상 되는 답변을 만들기 위해 참고한 context
- reference: 예상되는 답변

--------------------------

- user_input: 질문
- retrieved_contexts: 검색한 자료 -> 리스트
- response: 실제 답변

In [14]:
# for 문으로 구현하기
answer = []
contexts = []
for question in df['user_input']:
    docs = retriever.invoke(question)

    ctx_list = []
    for doc in docs:
        ctx_list.append(doc.page_content)
    contexts.append(ctx_list)


In [15]:
for question in df['user_input']:
    ans = rag_chain.invoke({"question": question})
    answer.append(ans)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4.1-mini in organization org-ifMd863rcCIXpEuEYtSj7yxy on tokens per min (TPM): Limit 200000, Used 200000, Requested 70. Please try again in 21ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}

In [ ]:
answer